In [ ]:
import matplotlib.pyplot as plt
import sys
from typing import List, Dict
if not '../..' in sys.path:
    sys.path.append('../..')
import basico
import numpy
import pandas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import random
import itertools
import math
import tqdm
import libsbml
import pickle
import time
import os

In [ ]:
fname = "../ParkRam_constant_feed.xlsx"
data = pandas.read_excel(fname, sheet_name=1)
data

In [ ]:
def convert_symbol_to_label(symbol):
    return f"{symbol}_noise"

def convert_symbol_to_sd(symbol):
    return f"sd_{symbol}"

In [ ]:
with open("../PR_testbatch.txt", "r") as f:
    testbatch_str = f.read().split()
    testbatch = set([int(el) for el in testbatch_str])

prev_val = 0
batches_lengths = []
for val in data["exp"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1

not_testbatch = set(range(len(batches_lengths))) - testbatch
not_testbatch_exp_labels = [el + 1 for el in not_testbatch]

In [ ]:
maxes = {}
mins = {}
concentrations_symbols = ["X", "S", "PT", "PM", "V"]
concentrations_labels = [convert_symbol_to_label(symbol) for symbol in concentrations_symbols]
sd_labels = [convert_symbol_to_sd(symbol) for symbol in concentrations_symbols]

for label in (['t'] + concentrations_labels):
    max_val = max(data[data["exp"].isin(not_testbatch_exp_labels)][label])
    min_val = min(data[data["exp"].isin(not_testbatch_exp_labels)][label])
    label_scaled = f"{label}_scaled"
    maxes[label] = max_val
    mins[label] = min_val
    data[label_scaled] = (data[label] - min_val) / (max_val - min_val)

#sd scaling
for symbol in concentrations_symbols:
    label_sd = convert_symbol_to_sd(symbol)
    label_sd_scaled = f"{label_sd}_scaled"
    label_concentration = convert_symbol_to_label(symbol)
    data[label_sd_scaled] = data[label_sd] / (maxes[label_concentration] - mins[label_concentration])

In [ ]:
data

In [ ]:

batches = []
for i, length in enumerate(batches_lengths):
    batches.append(data.iloc[i*length : i*length + length].reset_index(drop=True))
length = batches_lengths[0]

#scale inputs
changing_vars = ["X_noise", "S_noise", "V_noise", "Feed"]
first_of_each_batch = data[changing_vars].iloc[::length]
mins_Xs = first_of_each_batch.min()
maxes_Xs = first_of_each_batch.max()
scaled_initial_conditions = (first_of_each_batch - mins_Xs)/(maxes_Xs-mins_Xs)

Xs_ys_sds = []
for i, batch in enumerate(batches):
    scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
    X = pandas.concat([batch[["t_scaled"]], scaled_initials_for_this_batch], axis=1)
    y = batch[[f"{label}_scaled" for label in concentrations_labels]]
    sd = batch[[f"{convert_symbol_to_sd(symbol)}_scaled" for symbol in concentrations_symbols]]
    X = torch.Tensor(X.values)
    y = torch.Tensor(y.values)
    sd = torch.Tensor(sd.values)
    Xs_ys_sds.append((X, y, sd))

In [ ]:
def wmse(input, target, sd):
    return torch.sum(((input - target) / sd) ** 2) / input.numel() / 100

loss_fn = wmse

In [ ]:
INPUTS = 5
OUTPUTS = 5
NUM_HIDDEN = 1
DROPOUT = 0
STOCH_REG_BATCH_SIZE = 1
LR = 0.0001

class PR_rnn_tanh(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(PR_rnn_tanh, self).__init__()
        
        # Number of hidden dimensions
        self.hidden_dim = hidden_dim
        
        # Number of hidden layers
        self.layer_dim = layer_dim
        
        # RNN
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True)
        
        # Readout layer
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(device='cuda')
            
        # One time step
        out, hn = self.rnn(x, h0)
        out = self.fc(out) 
        return out

In [ ]:
def AICc(nw, model, n_train, dataloader):
    for X, y, sd in dataloader:
        y_pred = model(X)
        Train_Loss = loss_fn(y_pred, y, sd)
        aic = n_train * math.log(Train_Loss) + 2*nw + 2*nw*(nw+1)/(n_train-nw-1)
        return aic

In [ ]:
n_epochs = 1000000
num_batches = len(batches_lengths)
nums_nodes_one_layer = [10, 15, 19]
num_repetitions = 3

with open("../PR_train_validation_test", "rb") as file:
    train_validation_test = pickle.load(file)
time_to_wait_for_improvement = 2000

In [ ]:
num_train_batches = len(train_validation_test[0][0])
example_y = Xs_ys_sds[0][1]
n_train = example_y.numel() * num_train_batches

In [ ]:
counter = 0
while os.path.exists(f"PR_rnn_tanh_{counter}.csv"):
    counter += 1
csv_name = f"PR_rnn_tanh_{counter}.csv"

In [ ]:
newpath = "saved_models"
if not os.path.exists(newpath):
    os.makedirs(newpath)

In [ ]:

csv_counter = 0
csv_max = 100
csv_first = True
columns = ['num_nodes_on_layer', 'start_weights_iteration', 'k_fold', 'epoch', 'train', 'validation']
nums_nodes_on_layer_for_csv = []
start_weights_iteration = []
k_folds = []
epochs = []
train_losses = []
validation_losses = []
for num_nodes_one_layer in nums_nodes_one_layer:
    best_validation_loss_model_global = None
    best_validation_loss_global = math.inf
    best_train_val_test_global = None
    for j in range(num_repetitions):
        nn_model = PR_rnn_tanh(INPUTS, num_nodes_one_layer, NUM_HIDDEN, OUTPUTS)
        number_of_parameters = sum(torch.numel(tensor) for tensor in nn_model.state_dict().values())
        models = []
        test_losses = []
        train_losses_end = []
        val_losses_end = []
        nums_epoch = []
        AICs = []
        ti = time.time()
        for i, (train_batches_nums, validation_batches_nums, test_batches_nums) in enumerate(train_validation_test):

            best_validation_loss_model = None
            best_validation_loss = math.inf
            
            train_batches = [Xs_ys_sds[i] for i in train_batches_nums]
            validation_batches = [Xs_ys_sds[i] for i in validation_batches_nums]
            test_batches = [Xs_ys_sds[i] for i in test_batches_nums]
            
            X_train = torch.stack([batch[0] for batch in train_batches])
            y_train = torch.stack([batch[1] for batch in train_batches])
            sd_train = torch.stack([batch[2] for batch in train_batches])
            X_validation = torch.stack([batch[0] for batch in validation_batches])
            y_validation = torch.stack([batch[1] for batch in validation_batches])
            sd_validation = torch.stack([batch[2] for batch in validation_batches])
            X_test = torch.stack([batch[0] for batch in test_batches])
            y_test = torch.stack([batch[1] for batch in test_batches])
            sd_test = torch.stack([batch[2] for batch in test_batches])
            
            X_train = X_train.to(device='cuda')
            y_train = y_train.to(device='cuda')
            sd_train = sd_train.to(device='cuda')
            X_validation = X_validation.to(device='cuda')
            y_validation = y_validation.to(device='cuda')
            sd_validation = sd_validation.to(device='cuda')
            X_test = X_test.to(device='cuda')
            y_test = y_test.to(device='cuda')
            sd_test = sd_test.to(device='cuda')
            
            training_data = TensorDataset(X_train, y_train, sd_train)
            validation_data = TensorDataset(X_validation, y_validation, sd_validation)
            test_data = TensorDataset(X_test, y_test, sd_test)
            
            train_dataloader = DataLoader(training_data, batch_size=STOCH_REG_BATCH_SIZE, shuffle=True)
            validation_dataloader = DataLoader(validation_data, batch_size=len(validation_data))
            test_dataloader = DataLoader(test_data, batch_size=len(test_data))

            this_iteration_model = PR_rnn_tanh(INPUTS, num_nodes_one_layer, NUM_HIDDEN, OUTPUTS)
            this_iteration_model.load_state_dict(nn_model.state_dict())
            this_iteration_model.to(device='cuda')
            optimizer = optim.Adam(this_iteration_model.parameters(), lr=LR, )
            counter = time_to_wait_for_improvement
            
            for epoch in range(n_epochs):
                this_iteration_model.train()
                for X, y, sd in train_dataloader:
                    y_pred = this_iteration_model(X)
                    loss = loss_fn(y_pred, y, sd)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                print(f'Finished epoch {epoch}, latest loss {loss}')
                train_losses.append(float(loss))
                sum_loss = 0
                num_loss = 0
                with torch.no_grad():
                    this_iteration_model.eval()
                    for X, y, sd in validation_dataloader:
                        y_pred = this_iteration_model(X)
                        loss = loss_fn(y_pred, y, sd)
                        sum_loss += loss
                        num_loss += 1
                    mean_loss = float(sum_loss / num_loss)
                if mean_loss < best_validation_loss: #- minimal_improvement:
                    counter = time_to_wait_for_improvement
                    best_validation_loss = mean_loss
                    best_validation_loss_model = PR_rnn_tanh(INPUTS, num_nodes_one_layer, NUM_HIDDEN, OUTPUTS)
                    best_validation_loss_model.to(device='cuda')
                    best_validation_loss_model.load_state_dict(this_iteration_model.state_dict())
                counter -= 1
                if counter < 0:
                    nums_epoch.append(epoch)
                    break
                validation_losses.append(mean_loss)
                print("validation loss", mean_loss)

                nums_nodes_on_layer_for_csv.append(num_nodes_one_layer)
                start_weights_iteration.append(j)
                k_folds.append(i)
                epochs.append(epoch)
                csv_counter += 1
                if csv_counter == 100:
                    csv_counter = 0
                    df = pandas.DataFrame(zip(nums_nodes_on_layer_for_csv,
                                              start_weights_iteration,
                                              k_folds,
                                              epochs,
                                              train_losses,
                                              validation_losses), columns = columns)
                    if csv_first:
                        csv_first = False
                        df.to_csv(csv_name, index=False)
                    else:
                        df.to_csv(csv_name, mode='a', index=False, header=False)
                    nums_nodes_on_layer_for_csv = []
                    start_weights_iteration = []
                    k_folds = []
                    epochs = []
                    train_losses = []
                    validation_losses = []
            else:
                nums_epoch.append(n_epochs)

            #models.append(best_validation_loss_model)
            torch.save(best_validation_loss_model.state_dict(), f"saved_models/PR_rnn_tanh_wmse_{num_nodes_one_layer}_{j}_{i}")
            
            if best_validation_loss < best_validation_loss_global:
                best_validation_loss_global = best_validation_loss
                best_validation_loss_model_global = best_validation_loss_model
                best_train_val_test_global = [train_batches_nums, validation_batches_nums, test_batches_nums]

            sum_loss = 0
            num_loss = 0
            with torch.no_grad():
                sum_loss = 0
                num_loss = 0
                best_validation_loss_model.eval()
                for X, y, sd in test_dataloader:
                    y_pred = best_validation_loss_model(X)
                    loss = loss_fn(y_pred, y, sd)
                    sum_loss += loss
                    num_loss += 1
                mean_test_loss = float(sum_loss / num_loss)

                test_losses.append(mean_test_loss)
                sum_loss = 0
                num_loss = 0
                total_train_dataloader = DataLoader(training_data, len(training_data))
                for X, y, sd in total_train_dataloader:
                    y_pred = best_validation_loss_model(X)
                    Train_Loss = loss_fn(y_pred, y, sd)
                    sum_loss += Train_Loss
                    num_loss += 1
                mean_train_loss = float(sum_loss / num_loss)
                train_losses_end.append(mean_train_loss)

                sum_loss = 0
                num_loss = 0
                for X, y, sd in validation_dataloader:
                    y_pred = best_validation_loss_model(X)
                    Val_Loss = loss_fn(y_pred, y, sd)
                    sum_loss += Val_Loss
                    num_loss += 1
                mean_val_loss = float(sum_loss / num_loss)
                val_losses_end.append(mean_val_loss)
                AICs.append(AICc(number_of_parameters, best_validation_loss_model, n_train, total_train_dataloader))

        elapsed_time = time.time() - ti
        mean_test = numpy.mean(test_losses)
        sd_test = numpy.std(test_losses, ddof=1)
        mean_AIC = numpy.mean(AICs)
        sd_AIC = numpy.std(AICs, ddof=1)
        print(mean_test)
        print(sd_test)
        print(nums_epoch)
        with open("PR_tanh_rnn.txt", "a") as f:
            f.write(f"dropout {DROPOUT}, number of hidden layers {NUM_HIDDEN}, number of nodes on layer {num_nodes_one_layer} mean test loss {mean_test}, sd {sd_test}, number of parameters {number_of_parameters}, batch size {STOCH_REG_BATCH_SIZE}, learning rate {LR}\n")
            f.write(f"{test_losses}\n")
            f.write(f"mean AICc: {mean_AIC}, sd AICc: {sd_AIC}\n")
            f.write(f"AICcs: {AICs}\n")
            f.write(f'time {elapsed_time}\n')
            f.write(f"iterations {nums_epoch}\n")
            f.write(f"validation losses {val_losses_end}\n")
            f.write(f"train losses {train_losses_end}\n\n")
    torch.save(best_validation_loss_model_global.state_dict(), f"saved_models/PR_rnn_tanh_wmse_{num_nodes_one_layer}_best")
    with open(f'saved_models/PR_rnn_tanh_wmse_{num_nodes_one_layer}_best_train_validation_test.pkl', 'wb') as f:
        pickle.dump(best_train_val_test_global, f)
df = pandas.DataFrame(zip(nums_nodes_on_layer_for_csv,
                          start_weights_iteration,
                          k_folds,
                          epochs,
                          train_losses,
                          validation_losses), columns = columns)
if csv_first:
    csv_first = False
    df.to_csv(csv_name, index=False)
else:
    df.to_csv(csv_name, mode='a', index=False, header=False)